# 稀疏注意力：Sliding Window + Sparse Patterns

> Mistral / Longformer / BigBird 等模型的高效长序列方案。

## 背景
标准 Attention O(N^2) 对长序列不现实。稀疏注意力只让每个 token attend 到部分 token：
- **Sliding Window** (Mistral): 每个 token 只看附近 W 个 token
- **Dilated Sliding Window** (Longformer): 滑动窗口 + 间隔扩张，增加感受野
- **Global Attention** (BigBird/Longformer): 几个特殊 token(如 CLS) attend 到全体

## Sliding Window 实现
构造 band mask: 只有 |i-j| <= W 的位置保留，其余填 -inf。
与 KV Cache 结合：只 cache 最近 W 个 token 的 K/V。

## 考察点
- 稀疏 mask 构造 (torch.triu/tril)
- 稀疏度与精度 trade-off
- Sliding Window + Rotary PE 的组合效果


In [ ]:
import torch
import torch.nn.functional as F
import math

def sliding_window_attention(Q: int, K: int, V: int, window_size: int) -> torch.Tensor:
    """
    Sliding Window Attention: 每个 token 只看前后 window_size 范围内的 token。
    Q,K,V: [seq_len, dim]
    """
    seq_len, dim = Q.shape
    scale = 1.0 / math.sqrt(dim)
    scores = Q @ K.T * scale  # [seq_len, seq_len]
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool)
    for i in range(seq_len):
        left = max(0, i - window_size)
        right = min(seq_len, i + window_size + 1)
        mask[i, left:right] = False
    scores[mask] = -float("inf")
    attn = F.softmax(scores, dim=-1)
    return attn @ V

def global_sparse_attention(Q: int, K: int, V: int, window_size: int, global_tokens: int = 2) -> torch.Tensor:
    """
    Global + Sliding Window: 前 global_tokens 个 token 对所有 token 可见。
    Q,K,V: [seq_len, dim]
    """
    seq_len, dim = Q.shape
    scale = 1.0 / math.sqrt(dim)
    scores = Q @ K.T * scale
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool)
    for i in range(seq_len):
        left = max(0, i - window_size)
        right = min(seq_len, i + window_size + 1)
        mask[i, left:right] = False
        mask[i, :global_tokens] = False  # global tokens always visible
    scores[mask] = -float("inf")
    attn = F.softmax(scores, dim=-1)
    return attn @ V

# 验证
seq_len, dim = 16, 8
Q = K = V = torch.randn(seq_len, dim)
window = 3

out_sw = sliding_window_attention(Q, K, V, window)
out_full = F.softmax(Q @ K.T / math.sqrt(dim), dim=-1) @ V
assert out_sw.shape == out_full.shape
assert not torch.allclose(out_sw, out_full), "window attention should differ from full"
print(f"✅ SlidingWindow: shape={out_sw.shape}, 与 full attention 不同(符合预期)")

out_gs = global_sparse_attention(Q, K, V, window, global_tokens=2)
assert out_gs.shape == out_full.shape
assert not torch.allclose(out_gs, out_full)
assert not torch.allclose(out_gs, out_sw), "global+sliding should differ from sliding-only"
print(f"✅ GlobalSparse: shape={out_gs.shape}, 与 slidding-only 也不同(符合预期)")
